In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments
import torch
import zipfile
import os
import wandb
import json
from peft import LoraConfig, get_peft_model, TaskType
os.environ["WANDB_MODE"] = "disabled"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict
import torch

file_path = "/kaggle/input/all-subtask/train.xlsx"
df = pd.read_excel(file_path)
df = df.iloc[:, 0:2]
df.columns = ["subtask", "step"]
dataset = Dataset.from_pandas(df)
dataset = dataset.shuffle(seed=42)
dataset = dataset.train_test_split(test_size=0.111)
dataset = DatasetDict({
    "train": dataset["train"],
    "valid": dataset["test"]
})
print(len(dataset["train"]))
print(len(dataset["valid"]))
# In một ví dụ đầu tiên từ tập train
first_example = dataset["train"][0]

print("\n--- VÍ DỤ ĐẦU TIÊN TỪ TẬP TRAIN ---")
print(f"subtask: {first_example['subtask']}")
print(f"step: {first_example['step']}")

In [ ]:

model_name = "Salesforce/codet5-small"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
print(model)
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.01,
    target_modules=["q", "k", "v", "o"]
)

# 4. Gắn LoRA vào mô hình
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
model.to(device)
model.config.use_cache = False

In [ ]:
max_input_length = 128
max_target_length = 256
def filter_long(example):
    # tính độ dài trước khi padding/truncation
    input_ids = tokenizer.encode("action2json: " + example["subtask"], truncation=False)
    label_ids = tokenizer.encode(example["step"], truncation=False)
    return len(input_ids) <= max_input_length and len(label_ids) <= max_target_length

# Lọc dataset
dataset = dataset.filter(filter_long)
print(len(dataset["train"]))

def preprocess_function(examples):
    inputs = ["action2json: " + x for x in examples["subtask"]]
    targets = examples["step"]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True, padding=False)
    labels = tokenizer(targets, max_length=max_target_length, truncation=True, padding=False)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["subtask", "step"]
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model,pad_to_multiple_of=8)

In [ ]:
from transformers import TrainerCallback
class PredictionCallback(TrainerCallback):
    def __init__(self, tokenizer, sample_texts):
        self.tokenizer = tokenizer
        self.sample_texts = sample_texts

    def on_epoch_end(self, args, state, control, **kwargs):
        print("\n" + "="*50)
        print(f"DỰ ĐOÁN SAU EPOCH {state.epoch:.0f}")
        print("="*50)
        
        model = kwargs['model']
        model.eval() # Chuyển mô hình sang chế độ đánh giá
        
        with torch.no_grad():
            for i, text in enumerate(self.sample_texts):
                processed_text = "action2json: " + text
                inputs = self.tokenizer(processed_text, return_tensors="pt", max_length=512, truncation=True)
                inputs = {k: v.to(model.device) for k, v in inputs.items()}
                # Dự đoán với model.generate
                generated_ids = model.generate(
                    **inputs,
                    max_length=512,
                    num_beams=4,
                    early_stopping=True
                )
                
                # Decode kết quả
                decoded_text = self.tokenizer.decode(generated_ids[0], skip_special_tokens=True)
                
                print(f"[{i+1}] Input: {text}")
                print(f"    Output: {decoded_text}")
        
        model.train() # Chuyển mô hình về lại chế độ huấn luyện


In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./codet5-small-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=3,
    warmup_steps=300,   
    logging_dir="./logs",
    logging_steps=200,
    save_total_limit=2,
    fp16=True,
    optim='adamw_torch',
    lr_scheduler_type='cosine',
    dataloader_num_workers=4,
    gradient_checkpointing=False,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none"
)

# Khởi tạo class Callback với các câu mẫu
sample_texts_for_prediction = [
    "Xác minh rằng liên kết 'Giới thiệu' có đổ bóng khối với độ dịch chuyển 1px theo trục x, 1px theo trục y, độ mờ 2px và không giãn nở, với sắc thái 'xám nhạt' (rgb(200,200,200)).",
    "Xác minh rằng chỉ báo loại nguồn của âm thanh là 'MP3', mức âm lượng mặc định là 75%, và biểu tượng loa hiển thị trạng thái đại diện cho mức âm lượng 75%.",
    "Xác minh rằng mục 'Sản phẩm' có màu chữ #333333, cỡ chữ 17px, độ đậm chữ 500 và nằm cách mục 'Trang chủ' 20px về phía bên phải."
]
prediction_callback = PredictionCallback(tokenizer, sample_texts_for_prediction)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["valid"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=[prediction_callback]
)

In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()
trainer.train()

In [ ]:
adapter_dir = "codet5-small-lora/adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

try:
    merged_model = model.merge_and_unload()        # merge LoRA vào backbone
    full_dir = "./codet5-small-lora/merged"
    merged_model.save_pretrained(full_dir)         # đây mới là model full AutoModel load được
    tokenizer.save_pretrained(full_dir)
    print("✅ Saved merged full model to:", full_dir)
except Exception as e:
    print("❌ Skip merge (optional):", e)

base_dir    = "/kaggle/working/codet5-small-lora"
adapter_dir = os.path.join(base_dir, "adapter")
merged_dir  = os.path.join(base_dir, "merged")

def zip_dir(src_dir: str, zip_path: str, arc_prefix: str):
    """Zip toàn bộ src_dir, giữ nguyên cấu trúc, đặt vào arc_prefix trong file zip."""
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(src_dir):
            for f in files:
                abs_path = os.path.join(root, f)
                rel_path = os.path.relpath(abs_path, src_dir)
                zf.write(abs_path, os.path.join(arc_prefix, rel_path))

# 1) Zip adapter riêng
if os.path.isdir(adapter_dir):
    zip_adapter = "/kaggle/working/codet5-small-lora-adapter.zip"
    zip_dir(adapter_dir, zip_adapter, arc_prefix="codet5-small-lora/adapter")
    print(f"Zipped adapter to: {zip_adapter}")
else:
    print("Adapter folder not found, skip.")

# 2) Zip merged riêng
if os.path.isdir(merged_dir):
    zip_merged = "/kaggle/working/codet5-small-lora-merged.zip"
    zip_dir(merged_dir, zip_merged, arc_prefix="codet5-small-lora/merged")
    print(f"Zipped merged to: {zip_merged}")
else:
    print("Merged folder not found, skip.")

print("\nDONE")